In [1]:
# Jupyter Notebook: Notebook_3_Evaluation_and_Search.ipynb
# ==============================================================================
# --- 1. УСТАНОВКА И ИМПОРТЫ ---
# ==============================================================================
# !pip install torch torchvision numpy pandas timm tqdm scikit-learn plotly matplotlib pillow

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import re
import random
import timm
import torchvision.transforms as T
import trimesh
# --- Визуализация ---
import plotly.express as px
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# --- Вспомогательные для поиска ---
from scipy.spatial.distance import cdist

def load_stl_xyz_only(stl_path, num_points=1024, verbose=False):
    """
    (ИСПРАВЛЕНО) Загружает, сэмплирует, центрирует и нормализует STL с подробным выводом ошибок.
    Установите verbose=True для одного файла, чтобы увидеть детальную отладку.
    """
    try:
        # 1. Загрузка сетки
        mesh = trimesh.load(stl_path, process=True, force='mesh')
        if verbose: print(f"[{stl_path}] Шаг 1: Trimesh загрузил объект типа {type(mesh)}")

        # 2. Обработка сцены (если Trimesh загрузил сцену вместо одной сетки)
        if isinstance(mesh, trimesh.Scene):
            if verbose: print(f"[{stl_path}] -> Это сцена, объединяем геометрию...")
            mesh = mesh.dump(concatenate=True)
        
        # 3. Проверка на наличие вершин
        if not hasattr(mesh, 'vertices') or len(mesh.vertices) == 0:
            if verbose: print(f"[{stl_path}] ОШИБКА: Сетка пуста (нет вершин).")
            return None
        
        # 4. Проверка площади поверхности (важно!)
        # Если площадь очень мала или равна нулю, сэмплирование не удастся.
        if mesh.area < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Площадь поверхности сетки почти равна нулю ({mesh.area}). Невозможно сэмплировать.")
            return None

        # 5. Сэмплирование точек
        points, _ = trimesh.sample.sample_surface(mesh, num_points)
        if verbose: print(f"[{stl_path}] Шаг 2: Сэмплировано {len(points)} точек.")
        
        # 6. Проверка количества точек
        if len(points) < 1: # Даже если мы запросили много, должна быть хотя бы одна
             if verbose: print(f"[{stl_path}] ОШИБКА: Не удалось сэмплировать ни одной точки.")
             return None
        
        # 7. (БЕЗ ИЗМЕНЕНИЙ) Выравнивание количества точек до num_points
        if len(points) < num_points:
            indices = np.random.choice(len(points), num_points, replace=True)
        else:
            indices = np.random.choice(len(points), num_points, replace=False)
        points = points[indices]

        # 8. (БЕЗ ИЗМЕНЕНИЙ) Центрирование и нормализация
        centroid = np.mean(points, axis=0)
        points -= centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))

        # 9. Проверка вырожденности (если все точки в одном месте)
        if max_dist < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Облако точек вырождено (все точки в одной координате).")
            return None
        
        points /= max_dist
        if verbose: print(f"[{stl_path}] -> Успешно обработано!")
        
        return points.astype(np.float32)
        
    except Exception as e:
        # 10. Отлов всех остальных ошибок
        if verbose:
            import traceback
            print(f"[{stl_path}] КРИТИЧЕСКАЯ ОШИБКА: Произошло необработанное исключение.")
            traceback.print_exc() # Печатаем полный traceback для детальной отладки
        return None

class PairedStlImageDataset(Dataset):
    """(ПЕРЕРАБОТАНО) Датасет для структуры 1 STL -> 25 изображений."""
    def __init__(self, stl_root, image_root, num_points=4096, image_size=224):
        self.num_points = num_points
        self.stl_root = Path(stl_root)
        self.image_root = Path(image_root)
        all_stl_files = sorted([f for f in self.stl_root.rglob('*.stl') if f.is_file()])
        
        self.paired_files = []
        # Паттерн для поиска 4 цифр в имени файла
        four_digit_pattern = re.compile(r'(\d{4})')

        for stl_path in tqdm(all_stl_files, desc="Сопоставление файлов"):
            match = four_digit_pattern.search(stl_path.stem)
            if not match:
                continue
            
            stl_number = match.group(1) # Извлекаем 4-значный номер
            
            # Ищем все изображения, начинающиеся с этого номера
            # (например, '0001_00.png', '0001_01.png', ...)
            image_paths = sorted(self.image_root.glob(f"{stl_number}_*.png"))
            
            for image_path in image_paths:
                self.paired_files.append((stl_path, image_path))
        
        print(f"\nНайдено {len(all_stl_files)} STL файлов.")
        print(f"Создано {len(self.paired_files)} пар (STL, Изображение) для обучения.")
        
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.paired_files)

    def __getitem__(self, idx):
        stl_path, image_path = self.paired_files[idx]
        points = load_stl_xyz_only(stl_path, self.num_points)
        if points is None: return None
        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = self.image_transform(image)
        except Exception: return None
        return torch.from_numpy(points), image_tensor

def paired_collate_fn(batch):
    """(Без изменений) Collate функция для нового датасета."""
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return None, None
    points, images = zip(*batch)
    return torch.stack(points), torch.stack(images)

# --- 2. АРХИТЕКТУРА МОДЕЛЕЙ ---

# 2.1 STL ЭНКОДЕР (PointNet++ из вашего кода)

# (ИСПРАВЛЕНО) Вспомогательные функции для PointNet++ в читаемом и рабочем виде
def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long, device=device).view(1, 1, N).repeat(B, S, 1)
    sqrdists = torch.sum((xyz.unsqueeze(1) - new_xyz.unsqueeze(2)) ** 2, -1)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint, self.radius, self.nsample, self.group_all = npoint, radius, nsample, group_all
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last_channel = in_channel + 3
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        if not self.group_all:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            group_idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, group_idx)
            grouped_xyz -= new_xyz.unsqueeze(2)
            if points is not None:
                grouped_points = index_points(points, group_idx)
                features = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                features = grouped_xyz
        else:
            new_xyz = torch.zeros(xyz.shape[0], 1, 3, device=xyz.device)
            grouped_xyz = xyz.view(xyz.shape[0], 1, -1, 3)
            if points is not None:
                features = torch.cat([grouped_xyz, points.view(points.shape[0], 1, -1, points.shape[2])], dim=-1)
            else:
                features = grouped_xyz
        
        features = features.permute(0, 3, 2, 1)
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            features = F.relu(bn(conv(features)))
        
        new_points = torch.max(features, 2)[0].permute(0, 2, 1)
        return new_xyz, new_points

class StlEncoder(nn.Module):
    """(ИСПРАВЛЕНО) Ваш класс Encoder, переименован для ясности."""
    def __init__(self, in_features=3, embedding_dim=256):
        super().__init__()
        # in_channel теперь правильно 0, так как у нас нет доп. фичей кроме xyz
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=in_features-3, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=128, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=256, mlp=[256, 512, 1024], group_all=True)
        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc_embedding = nn.Linear(512, embedding_dim)
    
    def forward(self, xyz):
        l1_xyz, l1_points = self.sa1(xyz, points=None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        _, l3_points = self.sa3(l2_xyz, l2_points)
        x = l3_points.view(xyz.shape[0], -1)
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        embedding = self.fc_embedding(x)
        return F.normalize(embedding, dim=1)

class ViTImageEncoder(nn.Module):
    def __init__(self, embedding_dim=256, projection_dim=256): # projection_dim здесь не используется, но нужен для совпадения архитектуры
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        vit_embed_dim = self.backbone.embed_dim
        self.embedding_head = nn.Linear(vit_embed_dim, embedding_dim)
        self.projection_head = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim), nn.ReLU(), nn.Linear(embedding_dim, projection_dim)
        )
    def forward(self, x): # Для инференса нам нужен только чистый эмбеддинг
        features = self.backbone(x)
        embedding = self.embedding_head(features)
        projection = self.projection_head(embedding)
        return torch.nn.functional.normalize(projection, dim=1)

class StlEncoderWithProjection(nn.Module):
    def __init__(self, embedding_dim=256, projection_dim=256):
        super().__init__()
        self.backbone = StlEncoder(embedding_dim=embedding_dim)
        self.projection_head = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim), nn.ReLU(), nn.Linear(embedding_dim, projection_dim)
        )
    def forward(self, x): # Для инференса нам нужен только выход из backbone
        embedding = self.backbone(x)
        projection = self.projection_head(embedding)
        return torch.nn.functional.normalize(projection, dim=1)

print("Все библиотеки и классы моделей готовы.")


Все библиотеки и классы моделей готовы.


In [2]:
# ==============================================================================
# --- 2. КОНФИГУРАЦИЯ ---
# ==============================================================================
# --- ПУТИ К ДАННЫМ ---
TEST_STL_DIR = Path("test/test_data/gallery_mesh_for_image")
TEST_IMG_DIR = Path("test/test_data/gallery_image_for_mesh")

# --- ПУТЬ К ОБУЧЕННОЙ МОДЕЛИ ---
MODEL_CHECKPOINT_PATH = Path("supcon_vit_stl_final.pth")

# --- ПАРАМЕТРЫ (должны совпадать с параметрами обучения) ---
EMBEDDING_DIM = 256
IMAGE_SIZE = 224
NUM_POINTS = 4096
device = torch.device("cpu")

print(f"Используемое устройство: {device}")
print(f"Путь к модели: {MODEL_CHECKPOINT_PATH}")
print(f"Тестовые STL: {TEST_STL_DIR}")
print(f"Тестовые изображения: {TEST_IMG_DIR}")


Используемое устройство: cpu
Путь к модели: supcon_vit_stl_final.pth
Тестовые STL: test\test_data\gallery_mesh_for_image
Тестовые изображения: test\test_data\gallery_image_for_mesh


In [3]:
# ==============================================================================
# --- 3. ЗАГРУЗКА ОБУЧЕННЫХ МОДЕЛЕЙ ---
# ==============================================================================
print("\n--- Загрузка обученных моделей ---")

# Инициализируем модели с той же архитектурой, что и при обучении
image_encoder = ViTImageEncoder(embedding_dim=EMBEDDING_DIM).to(device)
stl_encoder = StlEncoderWithProjection(embedding_dim=EMBEDDING_DIM).to(device)

# Загружаем checkpoint
try:
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=device)
    image_encoder.load_state_dict(checkpoint['image_encoder_state_dict'])
    stl_encoder.load_state_dict(checkpoint['stl_encoder_state_dict'])
except FileNotFoundError:
    raise FileNotFoundError(f"Файл с весами модели не найден: {MODEL_CHECKPOINT_PATH}")

# ВАЖНО: Переводим модели в режим инференса
image_encoder.eval()
stl_encoder.eval()

print("Модели успешно загружены и переведены в режим оценки (eval).")



--- Загрузка обученных моделей ---


C:\Users\Computer\AppData\Local\Temp\ipykernel_9892\2371030516.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_locati

Модели успешно загружены и переведены в режим оценки (eval).


In [5]:
# ==============================================================================
# --- 4. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---
# ==============================================================================

# Функция рендеринга из предыдущего ноутбука
def render_stl_with_matplotlib(stl_path, size=200):
    # (Код этой функции без изменений)
    try:
        mesh = trimesh.load(stl_path, force='mesh')
        if not hasattr(mesh, 'faces') or len(mesh.faces) == 0: raise ValueError("No faces")
        fig = plt.figure(figsize=(size/100, size/100), dpi=100)
        ax = fig.add_subplot(111, projection='3d')
        ax.axis('off'); ax.grid(False); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
        ax.plot_trisurf(
            mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
            triangles=mesh.faces, color='lightblue', edgecolor='k', linewidth=0.1
        )
        max_range = np.array([mesh.vertices[:,i].max()-mesh.vertices[:,i].min() for i in range(3)]).max() / 2.0
        mid = [ (mesh.vertices[:,i].max()+mesh.vertices[:,i].min()) * 0.5 for i in range(3)]
        ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
        ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
        ax.set_zlim(mid[2] - max_range, mid[2] + max_range)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0, facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf)
    except Exception:
        return Image.new('RGB', (size, size), (255, 255, 255)) # Возвращаем белое изображение в случае ошибки

# Функции для генерации эмбеддингов
def generate_stl_embeddings(encoder, stl_dir):
    stl_files = sorted([f for f in stl_dir.rglob('*.stl') if f.is_file()])
    embeddings = {}
    with torch.no_grad():
        for path in tqdm(stl_files, desc="Генерация STL эмбеддингов"):
            points = load_stl_xyz_only(path, NUM_POINTS)
            if points is not None:
                points_tensor = torch.from_numpy(points).unsqueeze(0).to(device)
                embedding = encoder(points_tensor).squeeze(0).cpu()
                embeddings[str(path)] = embedding
    return embeddings

def generate_image_embeddings(encoder, image_dir):
    img_files = sorted([f for f in image_dir.rglob('*') if f.suffix.lower() in ['.png', '.jpg', '.jpeg']])
    # Трансформации для теста - БЕЗ АУГМЕНТАЦИЙ!
    transform = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    embeddings = {}
    with torch.no_grad():
        for path in tqdm(img_files, desc="Генерация Image эмбеддингов"):
            try:
                img = Image.open(path).convert("RGB")
                img_tensor = transform(img).unsqueeze(0).to(device)
                embedding = encoder(img_tensor).squeeze(0).cpu()
                embeddings[str(path)] = embedding
            except Exception: continue
    return embeddings

print("Вспомогательные функции готовы.")


Вспомогательные функции готовы.


In [6]:
# ==============================================================================
# --- 5. ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ ДЛЯ ТЕСТОВЫХ ДАННЫХ (ИСПРАВЛЕННАЯ ВЕРСИЯ) ---
# ==============================================================================
print("\n--- Генерация эмбеддингов для тестового набора данных ---")

# --- Шаг 1: Генерация STL эмбеддингов ---
stl_embeddings_dict = generate_stl_embeddings(stl_encoder, TEST_STL_DIR)

# --- ИСПРАВЛЕНИЕ: Проверка на пустой результат ---
if not stl_embeddings_dict:
    raise ValueError(f"Не удалось сгенерировать ни одного STL эмбеддинга. "
                     f"Проверьте, что путь '{TEST_STL_DIR}' корректен и содержит валидные STL файлы.")
else:
    print(f"Успешно сгенерировано {len(stl_embeddings_dict)} STL эмбеддингов.")

# --- Шаг 2: Генерация Image эмбеддингов ---
image_embeddings_dict = generate_image_embeddings(image_encoder, TEST_IMG_DIR)

# --- ИСПРАВЛЕНИЕ: Проверка на пустой результат ---
if not image_embeddings_dict:
    raise ValueError(f"Не удалось сгенерировать ни одного Image эмбеддинга. "
                     f"Проверьте, что путь '{TEST_IMG_DIR}' корректен и содержит изображения.")
else:
    print(f"Успешно сгенерировано {len(image_embeddings_dict)} Image эмбеддингов.")


# --- Шаг 3: Преобразование в матрицы (теперь безопасно) ---
stl_paths = list(stl_embeddings_dict.keys())
stl_matrix = torch.stack(list(stl_embeddings_dict.values())).numpy()

image_paths = list(image_embeddings_dict.keys())
image_matrix = torch.stack(list(image_embeddings_dict.values())).numpy()

print(f"\nДанные готовы для анализа.")


--- Генерация эмбеддингов для тестового набора данных ---


Генерация STL эмбеддингов:   0%|          | 0/100 [00:00<?, ?it/s]

Успешно сгенерировано 100 STL эмбеддингов.


Генерация Image эмбеддингов:   0%|          | 0/2600 [00:00<?, ?it/s]

Успешно сгенерировано 2600 Image эмбеддингов.

Данные готовы для анализа.


In [ ]:
# ==============================================================================
# --- 6. ВИЗУАЛИЗАЦИЯ t-SNE ---
# ==============================================================================
print("\n--- Запуск t-SNE для визуализации ---")

# Объединяем эмбеддинги и создаем метаданные
all_embeddings = np.vstack([stl_matrix, image_matrix])
metadata = []
group_id_pattern = re.compile(r'g?(\d{4,6})')

for path in stl_paths:
    match = group_id_pattern.search(Path(path).stem)
    group_id = f"model_{match.group(1)}" if match else "unknown"
    metadata.append({"path": Path(path).name, "type": "STL", "group_id": group_id})

for path in image_paths:
    match = group_id_pattern.search(Path(path).stem)
    group_id = f"model_{match.group(1)}" if match else "unknown"
    metadata.append({"path": Path(path).name, "type": "Image", "group_id": group_id})

# Запускаем t-SNE
tsne = TSNE(n_components=3, perplexity=30, n_iter=1000, random_state=42, init='pca', learning_rate='auto')
tsne_results = tsne.fit_transform(all_embeddings)

# Создаем DataFrame для Plotly
df_plot = pd.DataFrame(tsne_results, columns=['x', 'y', 'z'])
df_plot = pd.concat([df_plot, pd.DataFrame(metadata)], axis=1)

# Рисуем интерактивный график
fig = px.scatter_3d(
    df_plot, x='x', y='y', z='z', color='group_id', symbol='type',
    hover_data=['path'], title="3D t-SNE пространства эмбеддингов на тестовых данных"
)
fig.update_traces(marker=dict(size=4), selector=dict(mode='markers'))
fig.show()


In [ ]:
# ==============================================================================
# --- 7. ИНТЕРАКТИВНЫЕ ЗАПРОСЫ (ФИНАЛЬНАЯ, КОРРЕКТНАЯ ВЕРСИЯ) ---
# ==============================================================================

# --- ВАЖНО! "Магическая команда" для корректной работы в Jupyter ---
%matplotlib inline
import matplotlib.pyplot as plt
import trimesh
import numpy as np
from pathlib import Path


# --- ИСПРАВЛЕНИЕ: Новая функция, которая РИСУЕТ на готовой оси, а не создает свою фигуру ---
def draw_stl_on_ax(ax, stl_path):
    """
    Загружает STL и рисует его на ПЕРЕДАННОЙ оси `ax` из Matplotlib.
    Не создает и не показывает свою собственную фигуру.
    """
    try:
        # Отключаем фон и сетку для чистоты
        ax.axis('off')
        ax.grid(False)
        ax.set_facecolor('white')
        
        mesh = trimesh.load(stl_path, force='mesh')
        if not hasattr(mesh, 'faces') or len(mesh.faces) == 0:
            raise ValueError("Сетка не содержит граней.")

        # Рисуем сетку прямо на переданной оси
        ax.plot_trisurf(
            mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
            triangles=mesh.faces,
            color='lightblue',
            edgecolor='k',
            linewidth=0.1
        )
        
        # Автоматически подбираем масштаб, чтобы модель помещалась в кадр
        max_range = np.array([mesh.vertices[:,i].max()-mesh.vertices[:,i].min() for i in range(3)]).max() / 2.0
        mid = [ (mesh.vertices[:,i].max()+mesh.vertices[:,i].min()) * 0.5 for i in range(3)]
        ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
        ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
        ax.set_zlim(mid[2] - max_range, mid[2] + max_range)

    except Exception as e:
        # Если рендер не удался, пишем текст ошибки прямо на оси
        ax.text(0.5, 0.5, 'Render\nError', ha='center', va='center', fontsize=12, color='red')
        ax.axis('off')


# --- 7а. Запрос Image-to-Mesh ---
print("\n--- Демонстрация поиска: Image-to-Mesh ---")

# 1. Выбираем данные для запроса (без изменений)
query_image_path = random.choice(image_paths)
query_image_embedding = image_embeddings_dict[query_image_path]

# 2. Ищем 5 самых похожих STL (без изменений)
distances = cdist(query_image_embedding.unsqueeze(0).numpy(), stl_matrix, 'cosine')[0]
top5_indices = np.argsort(distances)[:5]
top5_stl_paths = [stl_paths[i] for i in top5_indices]
top5_distances = [distances[i] for i in top5_indices]

# 3. Визуализируем результат (НОВАЯ ЛОГИКА)
# Создаем ОДНУ фигуру и сетку из 6 ячеек
fig, axes = plt.subplots(1, 6, figsize=(20, 4), subplot_kw={'projection': '3d'})
fig.suptitle(f"Запрос Image-to-Mesh\nQuery: {Path(query_image_path).name}", fontsize=14)

# Размещаем изображение-запрос. Для 2D-картинки проекция '3d' не нужна, убираем ее.
axes[0].remove()
ax_query_2d = fig.add_subplot(1, 6, 1)
ax_query_2d.imshow(Image.open(query_image_path))
ax_query_2d.set_title("Query Image", fontsize=10)
ax_query_2d.axis('off')

# В цикле вызываем новую функцию, передавая ей нужную ячейку для рисования
for i, (stl_path, dist) in enumerate(zip(top5_stl_paths, top5_distances)):
    ax = axes[i+1] # Берем ячейку из нашей таблицы
    draw_stl_on_ax(ax, stl_path) # Рисуем прямо на ней
    ax.set_title(f"Top-{i+1}: {Path(stl_path).name}\nDist: {dist:.4f}", fontsize=10)

fig.subplots_adjust(top=0.7)
plt.show() # Показываем ОДНУ готовую фигуру


# --- 7б. Запрос Mesh-to-Image ---
print("\n--- Демонстрация поиска: Mesh-to-Image ---")

# 1. Выбираем данные для запроса (без изменений)
query_stl_path = random.choice(stl_paths)
query_stl_embedding = stl_embeddings_dict[query_stl_path]

# 2. Ищем 5 самых похожих изображений (без изменений)
distances = cdist(query_stl_embedding.unsqueeze(0).numpy(), image_matrix, 'cosine')[0]
top5_indices = np.argsort(distances)[:5]
top5_image_paths = [image_paths[i] for i in top5_indices]
top5_distances = [distances[i] for i in top5_indices]

# 3. Визуализируем результат (НОВАЯ ЛОГИКА)
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
fig.suptitle(f"Запрос Mesh-to-Image\nQuery: {Path(query_stl_path).name}", fontsize=14)

# Рисуем STL-запрос в первой ячейке
# Для этого нужно сделать ее 3D-проекцией
axes[0].remove()
ax_query_3d = fig.add_subplot(1, 6, 1, projection='3d')
draw_stl_on_ax(ax_query_3d, query_stl_path)
ax_query_3d.set_title("Query STL", fontsize=10)

# Размещаем 5 изображений-результатов в остальных ячейках (они остаются 2D)
for i, (img_path, dist) in enumerate(zip(top5_image_paths, top5_distances)):
    axes[i+1].imshow(Image.open(img_path))
    axes[i+1].set_title(f"Top-{i+1}: {Path(img_path).name}\nDist: {dist:.4f}", fontsize=10)
    axes[i+1].axis('off')
    
fig.subplots_adjust(top=0.7)
plt.show()

In [ ]:
# ==============================================================================
# --- 8а. Систематическая оценка: Image-to-Mesh Queries ---
# ==============================================================================
print("\n--- Запуск систематической оценки для Image-to-Mesh ---")

# 1. Указываем папку с изображениями-запросами
QUERY_IMG_DIR = Path("test/test_data/queries_image_to_mesh")

# 2. Генерируем эмбеддинги ТОЛЬКО для этих запросов
print(f"Генерация эмбеддингов для запросов из: {QUERY_IMG_DIR}...")
query_img_embeddings_dict = generate_image_embeddings(image_encoder, QUERY_IMG_DIR)

# 3. Проверяем, что эмбеддинги сгенерировались
if not query_img_embeddings_dict:
    print(f"\nОШИБКА: Не удалось сгенерировать эмбеддинги для изображений в папке {QUERY_IMG_DIR}. Проверьте путь и файлы.")
else:
    print(f"Начинаем поиск для {len(query_img_embeddings_dict)} изображений-запросов...")
    
    # 4. Проходим по каждому изображению-запросу
    for query_path, query_embedding in tqdm(query_img_embeddings_dict.items(), desc="Обработка Image-запросов"):
        
        # Ищем 5 самых похожих STL из общей галереи
        distances = cdist(query_embedding.unsqueeze(0).numpy(), stl_matrix, 'cosine')[0]
        top5_indices = np.argsort(distances)[:5]
        top5_stl_paths = [stl_paths[i] for i in top5_indices]
        top5_distances = [distances[i] for i in top5_indices]

        # Визуализируем результат для текущего запроса
        fig, axes = plt.subplots(1, 6, figsize=(20, 4), subplot_kw={'projection': '3d'})
        fig.suptitle(f"Запрос Image-to-Mesh\nQuery: {Path(query_path).name}", fontsize=14)

        # Размещаем изображение-запрос
        axes[0].remove()
        ax_query_2d = fig.add_subplot(1, 6, 1)
        ax_query_2d.imshow(Image.open(query_path))
        ax_query_2d.set_title("Query Image", fontsize=10)
        ax_query_2d.axis('off')

        # Выводим топ-5 рендеров
        for i, (stl_path, dist) in enumerate(zip(top5_stl_paths, top5_distances)):
            ax = axes[i+1]
            draw_stl_on_ax(ax, stl_path)
            ax.set_title(f"Top-{i+1}: {Path(stl_path).name}\nDist: {dist:.4f}", fontsize=10)

        fig.subplots_adjust(top=0.7)
        plt.show()

In [ ]:
# ==============================================================================
# --- 8б. Систематическая оценка: Mesh-to-Image Queries ---
# ==============================================================================
print("\n--- Запуск систематической оценки для Mesh-to-Image ---")

# 1. Указываем папку с STL-запросами
QUERY_STL_DIR = Path("test/test_data/queries_mesh_to_image")

# 2. Генерируем эмбеддинги ТОЛЬКО для этих запросов
print(f"Генерация эмбеддингов для запросов из: {QUERY_STL_DIR}...")
query_stl_embeddings_dict = generate_stl_embeddings(stl_encoder, QUERY_STL_DIR)

# 3. Проверяем, что эмбеддинги сгенерировались
if not query_stl_embeddings_dict:
    print(f"\nОШИБКА: Не удалось сгенерировать эмбеддинги для STL в папке {QUERY_STL_DIR}. Проверьте путь и файлы.")
else:
    print(f"Начинаем поиск для {len(query_stl_embeddings_dict)} STL-запросов...")

    # 4. Проходим по каждому STL-запросу
    for query_path, query_embedding in tqdm(query_stl_embeddings_dict.items(), desc="Обработка STL-запросов"):
        
        # Ищем 5 самых похожих изображений из общей галереи
        distances = cdist(query_embedding.unsqueeze(0).numpy(), image_matrix, 'cosine')[0]
        top5_indices = np.argsort(distances)[:5]
        top5_image_paths = [image_paths[i] for i in top5_indices]
        top5_distances = [distances[i] for i in top5_indices]

        # Визуализируем результат для текущего запроса
        fig, axes = plt.subplots(1, 6, figsize=(20, 4))
        fig.suptitle(f"Запрос Mesh-to-Image\nQuery: {Path(query_path).name}", fontsize=14)

        # Рисуем STL-запрос в первой ячейке
        axes[0].remove()
        ax_query_3d = fig.add_subplot(1, 6, 1, projection='3d')
        draw_stl_on_ax(ax_query_3d, query_path)
        ax_query_3d.set_title("Query STL", fontsize=10)

        # Выводим топ-5 изображений
        for i, (img_path, dist) in enumerate(zip(top5_image_paths, top5_distances)):
            axes[i+1].imshow(Image.open(img_path))
            axes[i+1].set_title(f"Top-{i+1}: {Path(img_path).name}\nDist: {dist:.4f}", fontsize=10)
            axes[i+1].axis('off')
            
        fig.subplots_adjust(top=0.7)
        plt.show()